Raw goal per minute for players

In [1]:
# === Load player data and build goals-per-90 (the raw, un-shrunk rate) ===
import pandas as pd
import numpy as np

BASE = r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot"
df = pd.read_parquet(BASE + r"\data\history\all_seasons_fixed.parquet")

# Aggregate to one row per player-season: total goals, total minutes
agg = (df[df["position"] != "AM"]                      # drop managers (bug #5)
       .groupby(["season", "element", "name", "position"])
       .agg(goals=("goals_scored", "sum"),
            minutes=("minutes", "sum"))
       .reset_index())

# Raw goals per 90
agg["raw_g90"] = agg["goals"] / agg["minutes"] * 90

# Look at one season, forwards only, to see the small-sample problem
fwd = agg[(agg["season"] == "2023-24") & (agg["position"] == "FWD")].copy()
print("Forwards in 2023-24:", len(fwd))
print("\nHighest raw goals/90 (watch the low-minute players):")
print(fwd.sort_values("raw_g90", ascending=False)
      [["name","minutes","goals","raw_g90"]].head(10).to_string(index=False))

Forwards in 2023-24: 113

Highest raw goals/90 (watch the low-minute players):
                name  minutes  goals  raw_g90
         Alejo Véliz       45      1 2.000000
      Sasa Kalajdzic      158      2 1.139241
          Jhon Durán      461      5 0.976139
      Erling Haaland     2553     27 0.951821
       Kieffer Moore      100      1 0.900000
      Alexander Isak     2253     21 0.838881
       Callum Wilson      981      9 0.825688
          Chris Wood     1801     14 0.699611
      Elijah Adebayo     1405     10 0.640569
Jean-Philippe Mateta     2274     16 0.633245


Finding the variance and K

In [2]:
# === Compute the shrinkage ingredients for forwards, 2023-24 ===
# Require a minimum sample so the "average" isn't polluted by 20-minute cameos
pool = fwd[fwd["minutes"] >= 90].copy()   # at least one full match played
print("Forwards with >=90 mins:", len(pool))

# --- Ingredient 1: the prior (what we shrink toward) ---
# Minutes-weighted mean rate = total goals / total minutes * 90 (pooled)
league_g90 = pool["goals"].sum() / pool["minutes"].sum() * 90
print(f"\nLeague-average striker goals/90 (the prior): {league_g90:.3f}")

# --- Ingredient 2: the variance split (signal vs noise) ---
# Total observed variance in raw rates (minutes-weighted)
w_min = pool["minutes"] / pool["minutes"].sum()
total_var = np.average((pool["raw_g90"] - league_g90)**2, weights=w_min)

# Expected sampling-noise variance for a Poisson count:
# each player's rate has noise ~ league_rate / (minutes/90).
# average it across players (weighted)
noise_var = np.average(league_g90 / (pool["minutes"] / 90), weights=w_min)

# Signal = total - noise
signal_var = max(total_var - noise_var, 1e-9)

print(f"\nTotal observed variance : {total_var:.4f}")
print(f"Expected sampling noise  : {noise_var:.4f}")
print(f"True between-player signal: {signal_var:.4f}")

# --- k falls out: noise scale / signal variance, in minutes ---
k = league_g90 / signal_var
print(f"\nk (minutes needed for half-trust): {k:.0f}")

Forwards with >=90 mins: 58

League-average striker goals/90 (the prior): 0.422

Total observed variance : 0.0416
Expected sampling noise  : 0.0280
True between-player signal: 0.0136

k (minutes needed for half-trust): 31


Raw vs Shrunk results

In [3]:
# === Apply shrinkage: w = n/(n+k), then blend player rate with the prior ===
# n is in the SAME units as k — here, 90-minute blocks
pool["n90"] = pool["minutes"] / 90
pool["w"] = pool["n90"] / (pool["n90"] + k)
pool["shrunk_g90"] = pool["w"] * pool["raw_g90"] + (1 - pool["w"]) * league_g90

print("Shrinkage applied. Compare raw vs shrunk:\n")
show = pool.sort_values("raw_g90", ascending=False)[
    ["name", "minutes", "goals", "raw_g90", "w", "shrunk_g90"]
].head(12)
print(show.round(3).to_string(index=False))

Shrinkage applied. Compare raw vs shrunk:

                        name  minutes  goals  raw_g90     w  shrunk_g90
              Sasa Kalajdzic      158      2    1.139 0.053       0.460
                  Jhon Durán      461      5    0.976 0.141       0.500
              Erling Haaland     2553     27    0.952 0.477       0.675
               Kieffer Moore      100      1    0.900 0.035       0.438
              Alexander Isak     2253     21    0.839 0.446       0.608
               Callum Wilson      981      9    0.826 0.260       0.527
                  Chris Wood     1801     14    0.700 0.392       0.531
              Elijah Adebayo     1405     10    0.641 0.334       0.495
        Jean-Philippe Mateta     2274     16    0.633 0.448       0.517
          Christopher Nkunku      437      3    0.618 0.135       0.448
Carlos Vinícius Alves Morais      306      2    0.588 0.099       0.438
                   Enes Ünal      317      2    0.568 0.102       0.437


Loading the Understat Data as it has the same id for players accross seasons

In [4]:
# === Load Understat season aggregates (stable cross-season id) ===
us = pd.read_parquet(BASE + r"\data\history\understat_season_aggregates.parquet")

# Bug #2: numeric columns are stored as strings — cast on load
num_cols = ["games","time","goals","xG","assists","xA","shots","key_passes",
            "yellow_cards","red_cards","npg","npxG","xGChain","xGBuildup"]
for c in num_cols:
    us[c] = pd.to_numeric(us[c])

print("Shape:", us.shape)
print("Columns:", us.columns.tolist())
print("\nSeasons present:", sorted(us["understat_season"].unique()))

# THE key test: is 'id' stable across seasons? (the thing element failed)
a = us[us["understat_season"]=="2023"][["id","player_name"]]
b = us[us["understat_season"]=="2024"][["id","player_name"]]
merged = a.merge(b, on="id", suffixes=("_23","_24"))
same = (merged["player_name_23"] == merged["player_name_24"]).sum()
print(f"\nid matches across 2023->2024: {len(merged)}")
print(f"  same player (name matches): {same} / {len(merged)}")

Shape: (5343, 19)
Columns: ['id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'position', 'team_title', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'understat_season']

Seasons present: ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

id matches across 2023->2024: 352
  same player (name matches): 352 / 352


Trying to shrink on Xg and not Raw goals per 90 minutes

In [5]:
# === Correct shrinkage validation on Understat id (npxG/90 per position) ===
us["min90"] = us["time"] / 90

def shrink_us(season, position, stat, min_time=450):
    pool = us[(us["understat_season"]==season) & (us["position"].str.contains(position, na=False))
              & (us["time"]>=min_time)].copy()
    if len(pool) < 10:
        return None, None
    prior = pool[stat].sum() / pool["time"].sum() * 90
    wmin = pool["time"]/pool["time"].sum()
    raw = pool[stat]/pool["time"]*90
    total_var = np.average((raw - prior)**2, weights=wmin)
    noise_var = np.average(prior/(pool["time"]/90), weights=wmin)
    signal_var = max(total_var - noise_var, 1e-9)
    k = prior/signal_var
    n90 = pool["time"]/90
    w = n90/(n90+k)
    pool["raw"] = raw
    pool["shrunk"] = w*raw + (1-w)*prior
    return pool[["id","player_name","raw","shrunk"]], k

# Understat position labels: 'F','M','D','S' etc. — use broad letters
# Validate: predict 2024 from 2023, for Forwards ('F' appears in F/S/FW labels)
season_pairs = [("2022","2023"), ("2023","2024")]
print("npxG/90 shrinkage — cross-season correlation (keyed on stable Understat id):\n")

for pos_label, pos_name in [("F","Forwards"), ("M","Midfielders"), ("D","Defenders")]:
    allr, allw = [], []
    for s_tr, s_nx in season_pairs:
        rates, k = shrink_us(s_tr, pos_label, "npxG")
        if rates is None: continue
        nxt = us[(us["understat_season"]==s_nx) & (us["position"].str.contains(pos_label, na=False))
                 & (us["time"]>=450)].copy()
        nxt["actual"] = nxt["npxG"]/nxt["time"]*90
        m = rates.merge(nxt[["id","actual"]], on="id", how="inner")
        allr.append(m)
    if allr:
        M = pd.concat(allr)
        print(f"{pos_name:12s} (n={len(M):3d}):  raw {M['raw'].corr(M['actual']):.3f}   "
              f"shrunk {M['shrunk'].corr(M['actual']):.3f}")

npxG/90 shrinkage — cross-season correlation (keyed on stable Understat id):

Forwards     (n=114):  raw 0.675   shrunk 0.646
Midfielders  (n=230):  raw 0.691   shrunk 0.675
Defenders    (n=207):  raw 0.482   shrunk 0.368


In [6]:
# === What do we have for the theoretical noise model? ===
# We have per-season totals: xG, shots. E[Q] = xG/shots is gettable.
# But Var(Q) — variance of individual shot quality — needs shot-level data.
us["shots_per90"] = us["shots"] / us["time"] * 90
us["xg_per_shot"] = us["xG"] / us["shots"]

check = us[(us["understat_season"]=="2023") & (us["time"]>=450)]
print("Shots per 90 — distribution:")
print(check["shots_per90"].describe().round(2))
print("\nAverage xG per shot (E[Q]):", round(check["xg_per_shot"].mean(), 3))
print("\nDo we have shot-level data anywhere? (need Var of individual shot quality)")
print("Columns available:", us.columns.tolist())

Shots per 90 — distribution:
count    402.00
mean       1.26
std        1.06
min        0.00
25%        0.45
50%        0.99
75%        1.96
max        5.29
Name: shots_per90, dtype: float64

Average xG per shot (E[Q]): 0.114

Do we have shot-level data anywhere? (need Var of individual shot quality)
Columns available: ['id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'position', 'team_title', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'understat_season', 'min90', 'shots_per90', 'xg_per_shot']


Trying to measure noise in shots by comparing even GW with odd GW for players

In [7]:
# === Approach B: measure xG/90 noise empirically via split-half ===
# Use vaastav gameweek xG (2022+), split each player-season into odd/even GWs
gw = df[df["position"] != "AM"].copy()
gw["expected_goals"] = pd.to_numeric(gw["expected_goals"], errors="coerce")
gw = gw[gw["season"].isin(["2022-23","2023-24","2024-25"]) & gw["expected_goals"].notna()]

gw["half"] = (gw["GW"] % 2)   # odd/even gameweeks = two random-ish halves

split = (gw.groupby(["season","element","name","half"])
         .agg(xg=("expected_goals","sum"), minutes=("minutes","sum"))
         .reset_index())
split["xg90"] = split["xg"]/split["minutes"]*90

# pivot so each player-season has half-0 and half-1 rates side by side
piv = split.pivot_table(index=["season","element","name"], columns="half",
                        values=["xg90","minutes"]).dropna()
piv.columns = ["min0","min1","xg90_0","xg90_1"]
piv = piv[(piv["min0"]>=200)&(piv["min1"]>=200)]   # enough in each half

# The variance BETWEEN halves = sampling noise (same player, two samples)
diff = piv["xg90_0"] - piv["xg90_1"]
empirical_noise_var = (diff**2).mean() / 2   # /2 because diff of two noisy estimates
print(f"Players with both halves >=200 min: {len(piv)}")
print(f"Empirical sampling-noise variance in xG/90: {empirical_noise_var:.4f}")
print(f"\nFor comparison — the Poisson-count noise we WRONGLY used earlier was ~0.028")

Players with both halves >=200 min: 1187
Empirical sampling-noise variance in xG/90: 0.0053

For comparison — the Poisson-count noise we WRONGLY used earlier was ~0.028


k = noise/variance, and noise here we took as 0.0053, which was disastero for defenders, for the correlation

In [8]:
# === Re-run xG shrinkage with the EMPIRICAL noise (0.0053), not the Poisson formula ===
# k = noise / signal_var, but now noise comes from the measured split-half value.
# The signal_var (true between-player spread) is still measured per position/season.

EMPIRICAL_NOISE = 0.0053   # from split-half, in xG/90 units

def shrink_us_empirical(season, position, stat, noise_var, min_time=450):
    pool = us[(us["understat_season"]==season) & (us["position"].str.contains(position, na=False))
              & (us["time"]>=min_time)].copy()
    if len(pool) < 10:
        return None, None
    prior = pool[stat].sum() / pool["time"].sum() * 90
    wmin = pool["time"]/pool["time"].sum()
    raw = pool[stat]/pool["time"]*90
    total_var = np.average((raw - prior)**2, weights=wmin)
    signal_var = max(total_var - noise_var, 1e-9)
    # k in 90-block units: noise scales per-90-block, so k = noise_per_block / signal
    # noise for a player with n90 blocks = noise_var_per_block / n90; matching the n/(n+k) form:
    k = noise_var / signal_var * np.average(pool["time"]/90, weights=wmin)  # calibrate to avg sample
    n90 = pool["time"]/90
    w = n90/(n90+k)
    pool["raw"] = raw
    pool["shrunk"] = w*raw + (1-w)*prior
    return pool[["id","player_name","raw","shrunk"]], k

print("npxG/90 shrinkage with EMPIRICAL noise — cross-season correlation:\n")
season_pairs = [("2022","2023"), ("2023","2024")]
for pos_label, pos_name in [("F","Forwards"), ("M","Midfielders"), ("D","Defenders")]:
    allr = []
    kvals = []
    for s_tr, s_nx in season_pairs:
        rates, k = shrink_us_empirical(s_tr, pos_label, "npxG", EMPIRICAL_NOISE)
        if rates is None: continue
        kvals.append(k)
        nxt = us[(us["understat_season"]==s_nx) & (us["position"].str.contains(pos_label, na=False))
                 & (us["time"]>=450)].copy()
        nxt["actual"] = nxt["npxG"]/nxt["time"]*90
        m = rates.merge(nxt[["id","actual"]], on="id", how="inner")
        allr.append(m)
    if allr:
        M = pd.concat(allr)
        print(f"{pos_name:12s} (n={len(M):3d}, k≈{np.mean(kvals):.1f}):  "
              f"raw {M['raw'].corr(M['actual']):.3f}   shrunk {M['shrunk'].corr(M['actual']):.3f}")

npxG/90 shrinkage with EMPIRICAL noise — cross-season correlation:

Forwards     (n=114, k≈5.2):  raw 0.675   shrunk 0.674
Midfielders  (n=230, k≈11.3):  raw 0.691   shrunk 0.684
Defenders    (n=207, k≈128420488.4):  raw 0.482   shrunk -0.089


Calculating the noise and k seperately for per position 

In [9]:
# === Per-position empirical noise, then per-position k and w ===

# Step 1: measure split-half noise SEPARATELY for each position
def measure_noise(pos_label):
    g = gw[gw["position"] == pos_label].copy()  # vaastav position labels: FWD/MID/DEF/GK
    g["half"] = g["GW"] % 2
    sp = (g.groupby(["season","element","half"])
          .agg(xg=("expected_goals","sum"), minutes=("minutes","sum")).reset_index())
    sp["xg90"] = sp["xg"]/sp["minutes"]*90
    pv = sp.pivot_table(index=["season","element"], columns="half",
                        values=["xg90","minutes"]).dropna()
    pv.columns = ["min0","min1","x0","x1"]
    pv = pv[(pv["min0"]>=200)&(pv["min1"]>=200)]
    return ((pv["x0"]-pv["x1"])**2).mean()/2, len(pv)

# NOTE: gw uses vaastav positions (FWD/MID/DEF); us uses Understat (F/M/D)
print("Per-position empirical xG/90 noise (split-half):")
noise_by_pos = {}
for vpos, upos in [("FWD","F"), ("MID","M"), ("DEF","D")]:
    nv, n = measure_noise(vpos)
    noise_by_pos[upos] = nv
    print(f"  {vpos}: noise = {nv:.4f}  (n={n})")

# Step 2: re-run shrinkage using each position's OWN noise
print("\nnpxG/90 shrinkage — per-position noise, k, w:\n")
for upos, pos_name in [("F","Forwards"), ("M","Midfielders"), ("D","Defenders")]:
    allr, kvals = [], []
    for s_tr, s_nx in [("2022","2023"), ("2023","2024")]:
        rates, k = shrink_us_empirical(s_tr, upos, "npxG", noise_by_pos[upos])
        if rates is None: continue
        kvals.append(k)
        nxt = us[(us["understat_season"]==s_nx) & (us["position"].str.contains(upos, na=False))
                 & (us["time"]>=450)].copy()
        nxt["actual"] = nxt["npxG"]/nxt["time"]*90
        allr.append(rates.merge(nxt[["id","actual"]], on="id", how="inner"))
    M = pd.concat(allr)
    print(f"{pos_name:12s} (n={len(M):3d}, noise={noise_by_pos[upos]:.4f}, k≈{np.mean(kvals):.1f}):  "
          f"raw {M['raw'].corr(M['actual']):.3f}   shrunk {M['shrunk'].corr(M['actual']):.3f}")

Per-position empirical xG/90 noise (split-half):
  FWD: noise = 0.0195  (n=131)
  MID: noise = 0.0059  (n=527)
  DEF: noise = 0.0016  (n=439)

npxG/90 shrinkage — per-position noise, k, w:

Forwards     (n=114, noise=0.0195, k≈47.4):  raw 0.675   shrunk 0.605
Midfielders  (n=230, noise=0.0059, k≈13.2):  raw 0.691   shrunk 0.681
Defenders    (n=207, noise=0.0016, k≈18.4):  raw 0.482   shrunk 0.520


FINAL RESULT AFTER GRIDSEARCH FOR K

In [10]:
# === OPTION 1: cross-season noise (season-to-season bounce, not within-season) ===
# Match same player across consecutive seasons via stable Understat id, measure the bounce.
def cross_season_noise(pos_label):
    diffs = []
    for s_tr, s_nx in [("2022","2023"),("2023","2024"),("2021","2022"),("2020","2021")]:
        a = us[(us["understat_season"]==s_tr)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=450)]
        b = us[(us["understat_season"]==s_nx)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=450)]
        a = a.assign(r=a["npxG"]/a["time"]*90)[["id","r"]]
        b = b.assign(r=b["npxG"]/b["time"]*90)[["id","r"]]
        m = a.merge(b, on="id", suffixes=("_a","_b"))
        diffs.append((m["r_a"]-m["r_b"]))
    d = pd.concat(diffs)
    return (d**2).mean()/2   # /2: bounce shared between two noisy season-estimates

print("OPTION 1 — cross-season noise vs the within-season noise we used:")
for upos, name in [("F","FWD"),("M","MID"),("D","DEF")]:
    cs = cross_season_noise(upos)
    print(f"  {name}: cross-season {cs:.4f}   vs within-season {noise_by_pos[upos]:.4f}")

# === OPTION 2: gridsearch k per position directly against next-season correlation ===
def corr_at_k(pos_label, k_val):
    allr = []
    for s_tr, s_nx in [("2022","2023"),("2023","2024")]:
        pool = us[(us["understat_season"]==s_tr)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=450)].copy()
        prior = pool["npxG"].sum()/pool["time"].sum()*90
        raw = pool["npxG"]/pool["time"]*90
        n90 = pool["time"]/90
        w = n90/(n90+k_val)
        pool["shrunk"] = w*raw + (1-w)*prior
        pool["id_"]=pool["id"]
        nxt = us[(us["understat_season"]==s_nx)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=450)].copy()
        nxt["actual"]=nxt["npxG"]/nxt["time"]*90
        allr.append(pool[["id","shrunk"]].merge(nxt[["id","actual"]],on="id"))
    M=pd.concat(allr)
    return M["shrunk"].corr(M["actual"])

print("\nOPTION 2 — best k per position (gridsearch against next-season correlation):")
for upos, name in [("F","FWD"),("M","MID"),("D","DEF")]:
    ks = [0, 2, 5, 10, 20, 40, 80]
    scores = [(k, corr_at_k(upos, k)) for k in ks]
    best = max(scores, key=lambda x: x[1])
    line = "  ".join(f"k={k}:{c:.3f}" for k,c in scores)
    print(f"  {name}: {line}")
    print(f"       -> best k={best[0]} (corr {best[1]:.3f});  k=0 means NO shrinkage\n")

OPTION 1 — cross-season noise vs the within-season noise we used:
  FWD: cross-season 0.0092   vs within-season 0.0195
  MID: cross-season 0.0045   vs within-season 0.0059
  DEF: cross-season 0.0016   vs within-season 0.0016

OPTION 2 — best k per position (gridsearch against next-season correlation):
  FWD: k=0:0.675  k=2:0.676  k=5:0.675  k=10:0.669  k=20:0.653  k=40:0.618  k=80:0.542
       -> best k=2 (corr 0.676);  k=0 means NO shrinkage

  MID: k=0:0.691  k=2:0.694  k=5:0.693  k=10:0.687  k=20:0.673  k=40:0.650  k=80:0.606
       -> best k=2 (corr 0.694);  k=0 means NO shrinkage

  DEF: k=0:0.482  k=2:0.492  k=5:0.502  k=10:0.513  k=20:0.521  k=40:0.522  k=80:0.505
       -> best k=40 (corr 0.522);  k=0 means NO shrinkage



SAME THING FOR ASSISTS NOW!

In [16]:
# === xA/90 shrinkage — same method, direct k-gridsearch per position ===
def corr_at_k_stat(pos_label, k_val, stat):
    allr = []
    for s_tr, s_nx in [("2022","2023"),("2023","2024")]:
        pool = us[(us["understat_season"]==s_tr)&(us["position"].str.contains(pos_label,na=False))
                  &(us["time"]>=450)].copy()
        prior = pool[stat].sum()/pool["time"].sum()*90
        raw = pool[stat]/pool["time"]*90
        n90 = pool["time"]/90
        w = n90/(n90+k_val)
        pool["shrunk"] = w*raw + (1-w)*prior
        nxt = us[(us["understat_season"]==s_nx)&(us["position"].str.contains(pos_label,na=False))
                 &(us["time"]>=450)].copy()
        nxt["actual"]=nxt[stat]/nxt["time"]*90
        allr.append(pool[["id","shrunk"]].merge(nxt[["id","actual"]],on="id"))
    M=pd.concat(allr)
    return M["shrunk"].corr(M["actual"]), len(M)

print("xA/90 shrinkage — best k per position (gridsearch vs next-season correlation):\n")
for upos, name in [("F","FWD"),("M","MID"),("D","DEF")]:
    ks = [0, 2, 5, 10, 20, 40, 80]
    scores = [(k, corr_at_k_stat(upos, k, "xA")[0]) for k in ks]
    n = corr_at_k_stat(upos, 0, "xA")[1]
    best = max(scores, key=lambda x: x[1])
    line = "  ".join(f"k={k}:{c:.3f}" for k,c in scores)
    print(f"  {name} (n={n}): {line}")
    print(f"       -> best k={best[0]} (corr {best[1]:.3f})\n")

xA/90 shrinkage — best k per position (gridsearch vs next-season correlation):

  FWD (n=114): k=0:0.512  k=2:0.511  k=5:0.507  k=10:0.495  k=20:0.469  k=40:0.415  k=80:0.320
       -> best k=0 (corr 0.512)

  MID (n=230): k=0:0.667  k=2:0.673  k=5:0.675  k=10:0.673  k=20:0.663  k=40:0.639  k=80:0.589
       -> best k=5 (corr 0.675)

  DEF (n=207): k=0:0.710  k=2:0.719  k=5:0.725  k=10:0.730  k=20:0.731  k=40:0.730  k=80:0.725
       -> best k=20 (corr 0.731)



In [17]:
# === Robustness: more season-pairs + does a round k match tuned k? ===

# Use ALL available consecutive season-pairs (Understat back to 2016)
all_pairs = [("2016","2017"),("2017","2018"),("2018","2019"),("2019","2020"),
             ("2020","2021"),("2021","2022"),("2022","2023"),("2023","2024")]

def corr_at_k_multi(pos_label, k_val, stat, pairs):
    allr = []
    for s_tr, s_nx in pairs:
        tr = us[(us["understat_season"]==s_tr)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=450)].copy()
        nx = us[(us["understat_season"]==s_nx)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=450)].copy()
        if len(tr)<10 or len(nx)<10: continue
        prior = tr[stat].sum()/tr["time"].sum()*90
        raw = tr[stat]/tr["time"]*90
        n90 = tr["time"]/90
        tr["shrunk"] = (n90/(n90+k_val))*raw + (1-(n90/(n90+k_val)))*prior
        nx["actual"] = nx[stat]/nx["time"]*90
        allr.append(tr[["id","shrunk"]].merge(nx[["id","actual"]],on="id"))
    M = pd.concat(allr)
    return M["shrunk"].corr(M["actual"]), len(M)

for stat in ["npxG","xA"]:
    print(f"\n===== {stat} — full history ({len(all_pairs)} season-pairs) =====")
    for upos, name in [("F","FWD"),("M","MID"),("D","DEF")]:
        ks = [0,2,5,10,20,40,80]
        scores = [(k,)+ (corr_at_k_multi(upos,k,stat,all_pairs),) for k in ks]
        # unpack
        scored = [(k, c) for (k,(c,n)) in [(s[0],(s[1])) for s in [(k, corr_at_k_multi(upos,k,stat,all_pairs)) for k in ks]]]
        n = corr_at_k_multi(upos,0,stat,all_pairs)[1]
        best = max(scored, key=lambda x:x[1])
        # compare tuned-best vs a single round k=10
        round_k = dict(scored)[10]
        line = "  ".join(f"k={k}:{c:.3f}" for k,c in scored)
        print(f"  {name} (n={n}): {line}")
        print(f"     best k={best[0]} ({best[1]:.3f})   |  round k=10 -> {round_k:.3f}   "
              f"(gap {best[1]-round_k:+.3f})")


===== npxG — full history (8 season-pairs) =====
  FWD (n=504): k=0:0.681  k=2:0.681  k=5:0.679  k=10:0.672  k=20:0.659  k=40:0.631  k=80:0.572
     best k=2 (0.681)   |  round k=10 -> 0.672   (gap +0.009)
  MID (n=966): k=0:0.744  k=2:0.748  k=5:0.748  k=10:0.746  k=20:0.738  k=40:0.718  k=80:0.671
     best k=5 (0.748)   |  round k=10 -> 0.746   (gap +0.003)
  DEF (n=857): k=0:0.462  k=2:0.470  k=5:0.477  k=10:0.483  k=20:0.485  k=40:0.477  k=80:0.445
     best k=20 (0.485)   |  round k=10 -> 0.483   (gap +0.002)

===== xA — full history (8 season-pairs) =====
  FWD (n=504): k=0:0.521  k=2:0.524  k=5:0.526  k=10:0.525  k=20:0.519  k=40:0.502  k=80:0.460
     best k=5 (0.526)   |  round k=10 -> 0.525   (gap +0.001)
  MID (n=966): k=0:0.675  k=2:0.681  k=5:0.685  k=10:0.687  k=20:0.684  k=40:0.673  k=80:0.639
     best k=10 (0.687)   |  round k=10 -> 0.687   (gap +0.000)
  DEF (n=857): k=0:0.696  k=2:0.705  k=5:0.710  k=10:0.713  k=20:0.710  k=40:0.699  k=80:0.666
     best k=10 (0.71

In [18]:
# === Honest test: pick k on EARLY seasons, evaluate on LATE seasons ===
early_pairs = [("2016","2017"),("2017","2018"),("2018","2019"),("2019","2020")]
late_pairs  = [("2020","2021"),("2021","2022"),("2022","2023"),("2023","2024")]

def best_k_on(pairs, pos_label, stat, ks=(0,2,5,10,20,40,80)):
    scores = {k: corr_at_k_multi(pos_label, k, stat, pairs)[0] for k in ks}
    return max(scores, key=scores.get), scores

print("Does the k chosen on EARLY seasons still win on LATE seasons?\n")
for stat in ["npxG","xA"]:
    print(f"===== {stat} =====")
    for upos, name in [("F","FWD"),("M","MID"),("D","DEF")]:
        # 1. pick best k using ONLY early seasons
        k_early, _ = best_k_on(early_pairs, upos, stat)
        # 2. what's the best k on late seasons (the "oracle")
        k_late, late_scores = best_k_on(late_pairs, upos, stat)
        # 3. how does early-chosen k perform on late? vs the late-oracle? vs k=10?
        perf_earlyk_on_late = late_scores[k_early]
        perf_oracle         = late_scores[k_late]
        perf_k10            = late_scores[10]
        print(f"  {name}: early picked k={k_early:<2d} -> scores {perf_earlyk_on_late:.3f} on late season")
        print(f"         late-oracle k={k_late:<2d} -> {perf_oracle:.3f}   |   universal k=10 -> {perf_k10:.3f}")
    print()

Does the k chosen on EARLY seasons still win on LATE seasons?

===== npxG =====
  FWD: early picked k=2  -> scores 0.657 on late season
         late-oracle k=0  -> 0.657   |   universal k=10 -> 0.651
  MID: early picked k=2  -> scores 0.708 on late season
         late-oracle k=5  -> 0.708   |   universal k=10 -> 0.704
  DEF: early picked k=10 -> scores 0.461 on late season
         late-oracle k=20 -> 0.461   |   universal k=10 -> 0.461

===== xA =====
  FWD: early picked k=5  -> scores 0.523 on late season
         late-oracle k=5  -> 0.523   |   universal k=10 -> 0.521
  MID: early picked k=5  -> scores 0.680 on late season
         late-oracle k=10 -> 0.682   |   universal k=10 -> 0.682
  DEF: early picked k=5  -> scores 0.733 on late season
         late-oracle k=10 -> 0.736   |   universal k=10 -> 0.736



FINISHING SKILL OF STRIKERS 

In [19]:
# === Does finishing skill (goals - xG) persist season to season? ===
# Finishing = actual goals minus expected goals, per 90. Positive = over-finisher.
# Use non-penalty to be clean: npg (non-pen goals) - npxG.

us["finishing90"] = (us["npg"] - us["npxG"]) / us["time"] * 90

def finishing_persistence(pos_label, min_time=900):
    # require decent minutes both seasons — finishing needs sample to even estimate
    allr = []
    for s_tr, s_nx in [("2016","2017"),("2017","2018"),("2018","2019"),("2019","2020"),
                       ("2020","2021"),("2021","2022"),("2022","2023"),("2023","2024")]:
        a = us[(us["understat_season"]==s_tr)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=min_time)]
        b = us[(us["understat_season"]==s_nx)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=min_time)]
        a = a[["id","finishing90"]].rename(columns={"finishing90":"fin_a"})
        b = b[["id","finishing90"]].rename(columns={"finishing90":"fin_b"})
        allr.append(a.merge(b, on="id"))
    M = pd.concat(allr)
    return M["fin_a"].corr(M["fin_b"]), len(M)

print("Does a player's finishing (npg - npxG per 90) predict next season's?\n")
for upos, name in [("F","FWD"),("M","MID"),("D","DEF")]:
    r, n = finishing_persistence(upos)
    print(f"  {name} (n={n}):  season-to-season correlation = {r:.3f}")

# For context: compare to how well npxG itself persists (the "chance quality" skill)
print("\nFor comparison — npxG/90 persistence (chance-generation skill):")
for upos, name in [("F","FWD"),("M","MID"),("D","DEF")]:
    r, n = corr_at_k_multi(upos, 0, "npxG",
            [("2016","2017"),("2017","2018"),("2018","2019"),("2019","2020"),
             ("2020","2021"),("2021","2022"),("2022","2023"),("2023","2024")])
    print(f"  {name}:  {r:.3f}")

Does a player's finishing (npg - npxG per 90) predict next season's?

  FWD (n=418):  season-to-season correlation = 0.190
  MID (n=780):  season-to-season correlation = 0.070
  DEF (n=672):  season-to-season correlation = -0.065

For comparison — npxG/90 persistence (chance-generation skill):
  FWD:  0.681
  MID:  0.744
  DEF:  0.462


Bucketing by price tier

In [20]:
# === Does finishing persistence concentrate in EXPENSIVE forwards? ===
# Attach price (name-match, per season) to forwards, split by price tier, check persistence.

fwd_pairs = [("2016","2017"),("2017","2018"),("2018","2019"),("2019","2020"),
             ("2020","2021"),("2021","2022"),("2022","2023"),("2023","2024")]

def finishing_by_price(min_time=900):
    allr = []
    for s_tr, s_nx in fwd_pairs:
        vseason = season_map.get(s_tr)
        if vseason is None:      # season_map only covers recent; extend it
            continue
        a = attach_price(s_tr)
        a = a[(a["position"].str.contains("F",na=False)) & (a["time"]>=min_time) & a["value"].notna()].copy()
        a["fin_a"] = (a["npg"]-a["npxG"])/a["time"]*90
        b = us[(us["understat_season"]==s_nx)&(us["position"].str.contains("F",na=False))&(us["time"]>=min_time)].copy()
        b["fin_b"] = (b["npg"]-b["npxG"])/b["time"]*90
        m = a[["id","value","fin_a"]].merge(b[["id","fin_b"]], on="id")
        allr.append(m)
    return pd.concat(allr)

# season_map currently only has 2020-2024; extend it for all seasons
season_map = {str(y): f"{y}-{str(y+1)[2:]}" for y in range(2016, 2025)}

fp = finishing_by_price()
print(f"Forwards with price + both seasons: {len(fp)}")

# Split into cheap / mid / premium by price
fp["tier"] = pd.qcut(fp["value"], 3, labels=["cheap","mid","premium"])
print("\nFinishing persistence by price tier (forwards):")
for tier in ["cheap","mid","premium"]:
    sub = fp[fp["tier"]==tier]
    print(f"  {tier:8s} (n={len(sub):3d}, avg £{sub['value'].mean()/10:.1f}m):  "
          f"corr = {sub['fin_a'].corr(sub['fin_b']):.3f}   "
          f"avg finishing = {sub['fin_a'].mean():+.3f}/90")

Forwards with price + both seasons: 172

Finishing persistence by price tier (forwards):
  cheap    (n= 58, avg £5.3m):  corr = 0.279   avg finishing = -0.018/90
  mid      (n= 57, avg £6.6m):  corr = 0.188   avg finishing = -0.020/90
  premium  (n= 57, avg £9.1m):  corr = 0.192   avg finishing = -0.037/90


Does the quality of the player carry forward?

In [22]:
# === Do shot volume and shot quality persist, and do they beat npxG? ===
pairs = [("2016","2017"),("2017","2018"),("2018","2019"),("2019","2020"),
         ("2020","2021"),("2021","2022"),("2022","2023"),("2023","2024")]

def persist(pos_label, col, min_time=900):
    allr = []
    for s_tr, s_nx in pairs:
        a = us[(us["understat_season"]==s_tr)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=min_time)]
        b = us[(us["understat_season"]==s_nx)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=min_time)]
        a = a[["id",col]].rename(columns={col:"a"}); b = b[["id",col]].rename(columns={col:"b"})
        allr.append(a.merge(b,on="id"))
    M = pd.concat(allr)
    return M["a"].corr(M["b"]), len(M)

print("Persistence season-to-season (how repeatable is each trait?):\n")
print(f"{'':12s} {'shots/90':>10s} {'xg/shot':>10s} {'npxG/90':>10s}")
for upos, name in [("F","FWD"),("M","MID"),("D","DEF")]:
    sv = persist(upos, "shots_per90")[0]
    sq = persist(upos, "xg_per_shot")[0]
    nx = persist(upos, "npxG")[0] if "npxG" in us else (None,)
    # npxG/90 persistence
    us["npxg90"] = us["npxG"]/us["time"]*90
    nx = persist(upos, "npxg90")[0]
    print(f"{name:12s} {sv:>10.3f} {sq:>10.3f} {nx:>10.3f}")

Persistence season-to-season (how repeatable is each trait?):

               shots/90    xg/shot    npxG/90
FWD               0.631      0.648      0.696
MID               0.811      0.584      0.776
DEF               0.722      0.236      0.496


Does adding shot volume help in predicting the xG?

In [23]:
# === Does adding shot volume improve GOAL prediction beyond npxG? ===
from sklearn.linear_model import LinearRegression

pairs = [("2016","2017"),("2017","2018"),("2018","2019"),("2019","2020"),
         ("2020","2021"),("2021","2022"),("2022","2023"),("2023","2024")]

def build(pos_label, min_time=900):
    rows = []
    for s_tr, s_nx in pairs:
        a = us[(us["understat_season"]==s_tr)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=min_time)].copy()
        b = us[(us["understat_season"]==s_nx)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=min_time)].copy()
        a["npxg90"]=a["npxG"]/a["time"]*90
        a["shots90"]=a["shots"]/a["time"]*90
        b["goals90"]=b["npg"]/b["time"]*90   # predict next-season non-pen goals/90
        rows.append(a[["id","npxg90","shots90"]].merge(b[["id","goals90"]],on="id"))
    return pd.concat(rows)

def cv_corr(X, y):
    # simple: fit on all, correlate fitted vs actual (in-sample proxy is fine for RELATIVE comparison)
    m = LinearRegression().fit(X, y)
    return np.corrcoef(m.predict(X), y)[0,1]

print("Predicting next-season goals/90 — does shot volume add to npxG?\n")
print(f"{'':12s} {'npxG only':>12s} {'npxG+shots':>12s} {'gain':>8s}")
for upos, name in [("F","FWD"),("M","MID"),("D","DEF")]:
    d = build(upos)
    c1 = cv_corr(d[["npxg90"]], d["goals90"])
    c2 = cv_corr(d[["npxg90","shots90"]], d["goals90"])
    print(f"{name:12s} {c1:>12.3f} {c2:>12.3f} {c2-c1:>+8.3f}   (n={len(d)})")

Predicting next-season goals/90 — does shot volume add to npxG?

                npxG only   npxG+shots     gain
FWD                 0.539        0.550   +0.011   (n=418)
MID                 0.668        0.683   +0.015   (n=780)
DEF                 0.374        0.443   +0.070   (n=672)


In [24]:
# === Held-out test: fit the model on EARLY seasons, evaluate on LATE seasons ===
early = [("2016","2017"),("2017","2018"),("2018","2019"),("2019","2020")]
late  = [("2020","2021"),("2021","2022"),("2022","2023"),("2023","2024")]

def build_pairs(pos_label, pairs, min_time=900):
    rows = []
    for s_tr, s_nx in pairs:
        a = us[(us["understat_season"]==s_tr)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=min_time)].copy()
        b = us[(us["understat_season"]==s_nx)&(us["position"].str.contains(pos_label,na=False))&(us["time"]>=min_time)].copy()
        a["npxg90"]=a["npxG"]/a["time"]*90
        a["shots90"]=a["shots"]/a["time"]*90
        b["goals90"]=b["npg"]/b["time"]*90
        rows.append(a[["id","npxg90","shots90"]].merge(b[["id","goals90"]],on="id"))
    return pd.concat(rows)

print("HONEST test — train on early seasons, predict on late seasons:\n")
print(f"{'':12s} {'npxG only':>12s} {'npxG+shots':>12s} {'gain':>8s}")
for upos, name in [("F","FWD"),("M","MID"),("D","DEF")]:
    tr = build_pairs(upos, early)
    te = build_pairs(upos, late)
    # model 1: npxG only
    m1 = LinearRegression().fit(tr[["npxg90"]], tr["goals90"])
    c1 = np.corrcoef(m1.predict(te[["npxg90"]]), te["goals90"])[0,1]
    # model 2: npxG + shots
    m2 = LinearRegression().fit(tr[["npxg90","shots90"]], tr["goals90"])
    c2 = np.corrcoef(m2.predict(te[["npxg90","shots90"]]), te["goals90"])[0,1]
    print(f"{name:12s} {c1:>12.3f} {c2:>12.3f} {c2-c1:>+8.3f}   (train {len(tr)}, test {len(te)})")

HONEST test — train on early seasons, predict on late seasons:

                npxG only   npxG+shots     gain
FWD                 0.494        0.495   +0.002   (train 204, test 214)
MID                 0.615        0.631   +0.016   (train 400, test 380)
DEF                 0.331        0.411   +0.081   (train 340, test 332)


Defensive Contributions

In [26]:
# === Load Core-Insights matchstats (defensive contribution source) ===
ms = pd.read_parquet(BASE + r"\data\history\core_insights_matchstats.parquet")

print("Shape:", ms.shape)
print("Seasons:", ms["season"].unique())

# Filter to Premier League only (drop CL/cups — they don't count for FPL)
ms_pl = ms[ms["match_id"].str.contains("-prem-", na=False)].copy()
print(f"\nPL-only rows: {len(ms_pl)} (dropped {len(ms)-len(ms_pl)} non-PL)")

# The defensive components you wanted
def_components = ["tackles","interceptions","recoveries","blocks",
                  "clearances","headed_clearances","defensive_contributions"]
print("\nComponent coverage (non-null %) by season:")
for c in def_components:
    ms_pl[c] = pd.to_numeric(ms_pl[c], errors="coerce")
print(ms_pl.groupby("season")[def_components].apply(lambda g: g.notna().mean().round(2)).to_string())

Shape: (26593, 66)
Seasons: <ArrowStringArray>
['2025-2026', '2024-2025']
Length: 2, dtype: str

PL-only rows: 24028 (dropped 2565 non-PL)

Component coverage (non-null %) by season:
           tackles  interceptions  recoveries  blocks  clearances  headed_clearances  defensive_contributions
season                                                                                                       
2024-2025      1.0            1.0         1.0     1.0         1.0                1.0                      0.0
2025-2026      1.0            1.0         1.0     1.0         1.0                1.0                      1.0


In [29]:
# === Attach position from the Core-Insights gameweek file (has player_id + position) ===
gwref = pd.read_parquet(BASE + r"\data\history\core_insights_gameweek_stats.parquet")
print("Gameweek file columns with id/position:",
      [c for c in gwref.columns if c in ["id","player_id","position","web_name","season"]])

# The gw file uses 'id' for player, 'position' as full word. Build a player_id -> position map.
pos_map = gwref[["id","position"]].drop_duplicates("id").set_index("id")["position"]
print("\nPositions available:", pos_map.unique())
print("Players with a position:", pos_map.notna().sum())

# Map onto matchstats
ms_pl["position"] = ms_pl["player_id"].map(pos_map)
print("\nMatchstats rows that got a position:", ms_pl["position"].notna().sum(), "of", len(ms_pl))
print("Unmatched:", ms_pl["position"].isna().sum())

Gameweek file columns with id/position: ['id', 'web_name', 'position', 'season']

Positions available: <ArrowStringArray>
['Goalkeeper', 'Defender', 'Midfielder', 'Forward']
Length: 4, dtype: str
Players with a position: 841

Matchstats rows that got a position: 24028 of 24028
Unmatched: 0


Defensive hit rates by position

In [30]:
# === Build DC score + hit-threshold target ===
ms_pl["cbit"]  = ms_pl["clearances"] + ms_pl["blocks"] + ms_pl["interceptions"] + ms_pl["tackles"]
ms_pl["cbirt"] = ms_pl["cbit"] + ms_pl["recoveries"]

ms_pl["is_def"] = ms_pl["position"] == "Defender"
ms_pl["dc_hit"] = np.where(ms_pl["is_def"],
                           (ms_pl["cbit"]  >= 10).astype(int),
                           (ms_pl["cbirt"] >= 12).astype(int))

played = ms_pl[ms_pl["minutes_played"] >= 1].copy()

print("DC threshold hit rate by position (players who appeared):\n")
for pos in ["Defender","Midfielder","Forward","Goalkeeper"]:
    sub = played[played["position"]==pos]
    print(f"  {pos:11s}: {sub['dc_hit'].mean():.1%}   (n={len(sub)}, avg CBIT {sub['cbit'].mean():.1f}, avg CBIRT {sub['cbirt'].mean():.1f})")

# Cross-check our computed score vs their column (2025-26 only)
chk = played[(played["season"]=="2025-2026")].copy()
chk["their_dc"] = pd.to_numeric(chk["defensive_contributions"], errors="coerce")
# their column should roughly equal cbit (def) or cbirt (others) — check correlation
print("\nSanity: our CBIT vs their defensive_contributions (2025-26):",
      round(chk["cbit"].corr(chk["their_dc"]), 3))

DC threshold hit rate by position (players who appeared):

  Defender   : 12.5%   (n=7441, avg CBIT 4.5, avg CBIRT 7.6)
  Midfielder : 13.6%   (n=10648, avg CBIT 3.0, avg CBIRT 6.1)
  Forward    : 5.8%   (n=2629, avg CBIT 2.0, avg CBIRT 4.1)
  Goalkeeper : 23.0%   (n=2048, avg CBIT 3.1, avg CBIRT 8.0)

Sanity: our CBIT vs their defensive_contributions (2025-26): nan


c:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Within season per position persistence 

In [36]:
# === WITHIN-season persistence: first half vs second half of same season ===
# Same player, same team, same season — no summer turnover. This is the fair test
# for actual FPL use (predict next GW from recent GWs).
components = ["tackles","interceptions","recoveries","blocks","clearances","headed_clearances"]

def within_season_persistence(position):
    rows_a, rows_b = [], []
    for season in ["2024-2025","2025-2026"]:
        s = played[(played["season"]==season) & (played["position"]==position)].copy()
        gw_med = s["gw"].median()
        first  = s[s["gw"] <= gw_med]
        second = s[s["gw"] >  gw_med]
        # per-90 rates in each half
        def rates(half):
            g = (half.groupby("player_id")
                 .agg(mins=("minutes_played","sum"),
                      **{c:(c,"sum") for c in components},
                      cbit=("cbit","sum"), cbirt=("cbirt","sum")).reset_index())
            g = g[g["mins"]>=270]  # ~3 full matches per half
            for c in components+["cbit","cbirt"]:
                g[c+"_90"]=g[c]/g["mins"]*90
            return g
        rows_a.append(rates(first)); rows_b.append(rates(second))
    A = pd.concat(rows_a); B = pd.concat(rows_b)
    out = {}
    for c in components+["cbit","cbirt"]:
        m = A[["player_id",c+"_90"]].merge(B[["player_id",c+"_90"]], on="player_id", suffixes=("_a","_b"))
        out[c] = (m[c+"_90_a"].corr(m[c+"_90_b"]), len(m))
    return out

positions = ["Defender","Midfielder","Forward"]
tables = {p: within_season_persistence(p) for p in positions}

print("WITHIN-season persistence (first half vs second half, same team):\n")
header = f"{'component':18s}" + "".join(f"{p[:3]:>10s}" for p in positions)
print(header); print("-"*len(header))
for c in components+["cbit","cbirt"]:
    row = f"{c:18s}" + "".join(f"{tables[p][c][0]:>10.3f}" for p in positions)
    print(row)
print(f"\n(n: DEF {tables['Defender']['cbit'][1]}, MID {tables['Midfielder']['cbit'][1]}, FWD {tables['Forward']['cbit'][1]})")

WITHIN-season persistence (first half vs second half, same team):

component                Def       Mid       For
------------------------------------------------
tackles                0.303     0.378     0.404
interceptions          0.296     0.379     0.344
recoveries             0.417     0.442     0.340
blocks                 0.372     0.386     0.445
clearances             0.464     0.510     0.703
headed_clearances      0.453     0.501     0.622
cbit                   0.443     0.512     0.570
cbirt                  0.426     0.483     0.422

(n: DEF 328, MID 443, FWD 100)


In [39]:
# === Are raw defensive actions systematically different between seasons? ===
components = ["tackles","interceptions","recoveries","blocks","clearances","headed_clearances"]

# Compare per-90 rates between seasons, defenders only, decent minutes
defs_all = played[(played["position"]=="Defender") & (played["minutes_played"]>=60)].copy()
for c in components + ["cbit"]:
    defs_all[c+"_90"] = defs_all[c] / defs_all["minutes_played"] * 90

print("Defender per-90 defensive actions by season (full-match appearances):\n")
print(f"{'component':18s} {'2024-25':>10s} {'2025-26':>10s} {'ratio':>8s}")
for c in components + ["cbit"]:
    a = defs_all[defs_all["season"]=="2024-2025"][c+"_90"].mean()
    b = defs_all[defs_all["season"]=="2025-2026"][c+"_90"].mean()
    print(f"{c:18s} {a:>10.2f} {b:>10.2f} {b/a:>8.2f}")

# Also check: is it minutes? maybe 2024-25 has more low-minute rows
print(f"\nAvg minutes/match — 2024-25: {played[played.season=='2024-2025']['minutes_played'].mean():.1f}"
      f"  2025-26: {played[played.season=='2025-2026']['minutes_played'].mean():.1f}")
print(f"Matches per season — 2024-25: {(played.season=='2024-2025').sum()}"
      f"  2025-26: {(played.season=='2025-2026').sum()}")

Defender per-90 defensive actions by season (full-match appearances):

component             2024-25    2025-26    ratio
tackles                  1.49       1.14     0.77
interceptions            0.68       1.04     1.52
recoveries               4.33       3.66     0.84
blocks                   0.29       0.57     1.94
clearances               1.97       4.37     2.21
headed_clearances        0.94       2.54     2.70
cbit                     4.44       7.11     1.60

Avg minutes/match — 2024-25: 64.9  2025-26: 65.4
Matches per season — 2024-25: 11567  2025-26: 11199


retrain each gameweek on all data so far, unlike xG CBIT is more week to week changing

In [42]:
# === Compare all 4 fixes for the drifting base rate (defenders, 2025-26) ===
from sklearn.isotonic import IsotonicRegression

feat = ["roll_dc90_3c","roll_dc90_5c","roll_hit_5","roll_mins_3"]
defs25 = d[(d["position"]=="Defender") & (d["season"]=="2025-2026")].dropna(subset=feat).copy().sort_values("gw")

def brier(y,p): return brier_score_loss(y, p)
def mk(): return lgb.LGBMClassifier(n_estimators=150, num_leaves=15, min_child_samples=40,
                                    learning_rate=0.05, random_state=42, verbose=-1)

# Evaluate everything on the SAME test window: GW 21..38, scoring each test GW.
test_gws = sorted([g for g in defs25["gw"].unique() if g >= 21])

# --- Fix 0: baseline (train once on GW<=20, no adaptation) ---
# --- Fix 1: same model + isotonic recalibration on a recent window ---
# --- Fix 2: add recent global hit-rate as a feature ---
# --- Fix 3: walk-forward retrain each test GW on all prior data ---
# --- Fix 4: recency-weighted training (recent matches weighted up) ---

# recent global hit-rate feature (rolling over prior GWs, all defenders)
gw_rate = defs25.groupby("gw")["dc_hit"].mean()
defs25["recent_base"] = defs25["gw"].map(
    {g: gw_rate[gw_rate.index < g].tail(3).mean() for g in defs25["gw"].unique()})

results = {f: {"y":[], "p":[]} for f in ["0_static","1_recalib","2_basefeat","3_walkfwd","4_recency"]}

# static model (fix 0/1 share it)
tr0 = defs25[defs25["gw"]<=20]
m0 = mk().fit(tr0[feat], tr0["dc_hit"])

# isotonic fit on a held-out slice of training (GW 15-20) for fix 1
cal_slice = defs25[(defs25["gw"]>=15)&(defs25["gw"]<=20)]
iso = IsotonicRegression(out_of_bounds="clip").fit(
    m0.predict_proba(cal_slice[feat])[:,1], cal_slice["dc_hit"])

for g in test_gws:
    te = defs25[defs25["gw"]==g]
    prior = defs25[defs25["gw"]<g]

    # 0 static
    p0 = m0.predict_proba(te[feat])[:,1]
    results["0_static"]["y"] += list(te["dc_hit"]); results["0_static"]["p"] += list(p0)
    # 1 recalibrated
    results["1_recalib"]["y"] += list(te["dc_hit"]); results["1_recalib"]["p"] += list(iso.predict(p0))
    # 2 base-rate feature
    m2 = mk().fit(tr0[feat+["recent_base"]].fillna(tr0["dc_hit"].mean()), tr0["dc_hit"])
    results["2_basefeat"]["y"] += list(te["dc_hit"]); results["2_basefeat"]["p"] += list(m2.predict_proba(te[feat+["recent_base"]].fillna(tr0["dc_hit"].mean()))[:,1])
    # 3 walk-forward retrain on all prior GWs
    if len(prior) > 200:
        m3 = mk().fit(prior[feat], prior["dc_hit"])
        p3 = m3.predict_proba(te[feat])[:,1]
        results["3_walkfwd"]["y"] += list(te["dc_hit"]); results["3_walkfwd"]["p"] += list(p3)
    # 4 recency-weighted (weight = 0.9^(gw_gap))
    w = 0.9 ** (g - prior["gw"].values)
    if len(prior) > 200:
        m4 = mk().fit(prior[feat], prior["dc_hit"], sample_weight=w)
        results["4_recency"]["y"] += list(te["dc_hit"]); results["4_recency"]["p"] += list(m4.predict_proba(te[feat])[:,1])

print("Fix comparison (test GW21-38):\n")
print(f"{'fix':14s} {'Brier':>8s} {'mean_pred':>10s} {'mean_actual':>12s}")
for f,r in results.items():
    if len(r["p"])==0: continue
    y,p = np.array(r["y"]), np.array(r["p"])
    print(f"{f:14s} {brier(y,p):>8.4f} {p.mean():>10.3f} {y.mean():>12.3f}")

Fix comparison (test GW21-38):

fix               Brier  mean_pred  mean_actual
0_static         0.1226      0.205        0.145
1_recalib        0.1457      0.212        0.145
2_basefeat       0.1201      0.202        0.145
3_walkfwd        0.1164      0.187        0.145
4_recency        0.1166      0.177        0.145


CBIRT for midfielders >= 12

In [43]:
# === Midfielder DC model (CBIRT >= 12) — walk-forward within 2025-26 ===
feat = ["roll_dc90_3c","roll_dc90_5c","roll_hit_5","roll_mins_3"]

mids25 = d[(d["position"]=="Midfielder") & (d["season"]=="2025-2026")].dropna(subset=feat).copy().sort_values("gw")
print(f"Midfielder matches 2025-26: {len(mids25)}  |  hit rate {mids25['dc_hit'].mean():.3f}")

def mk(): return lgb.LGBMClassifier(n_estimators=150, num_leaves=15, min_child_samples=40,
                                    learning_rate=0.05, random_state=42, verbose=-1)

test_gws = sorted([g for g in mids25["gw"].unique() if g >= 21])

# Compare the two winners from the defender analysis: static vs walk-forward vs recency
res = {k:{"y":[],"p":[]} for k in ["0_static","3_walkfwd","4_recency"]}
tr0 = mids25[mids25["gw"]<=20]
m0 = mk().fit(tr0[feat], tr0["dc_hit"])

for g in test_gws:
    te = mids25[mids25["gw"]==g]; prior = mids25[mids25["gw"]<g]
    res["0_static"]["y"]+=list(te["dc_hit"]); res["0_static"]["p"]+=list(m0.predict_proba(te[feat])[:,1])
    if len(prior)>200:
        m3=mk().fit(prior[feat],prior["dc_hit"])
        res["3_walkfwd"]["y"]+=list(te["dc_hit"]); res["3_walkfwd"]["p"]+=list(m3.predict_proba(te[feat])[:,1])
        w=0.9**(g-prior["gw"].values)
        m4=mk().fit(prior[feat],prior["dc_hit"],sample_weight=w)
        res["4_recency"]["y"]+=list(te["dc_hit"]); res["4_recency"]["p"]+=list(m4.predict_proba(te[feat])[:,1])

print("\nMidfielder fix comparison (test GW21+):\n")
print(f"{'fix':12s} {'Brier':>8s} {'mean_pred':>10s} {'mean_actual':>12s}")
for k,r in res.items():
    y,p=np.array(r["y"]),np.array(r["p"])
    base = np.full(len(y), tr0["dc_hit"].mean())
    print(f"{k:12s} {brier_score_loss(y,p):>8.4f} {p.mean():>10.3f} {y.mean():>12.3f}")
print(f"\nBaseline Brier (always base rate): {brier_score_loss(res['3_walkfwd']['y'], np.full(len(res['3_walkfwd']['y']), tr0['dc_hit'].mean())):.4f}")

# calibration for the walk-forward version
y,p = np.array(res["3_walkfwd"]["y"]), np.array(res["3_walkfwd"]["p"])
edges=np.arange(0,1.1,0.1); idx=np.clip(np.digitize(p,edges)-1,0,9)
print("\nWalk-forward calibration:")
for b in range(10):
    msk=idx==b
    if msk.sum()<15: continue
    print(f"  {edges[b]:.1f}-{edges[b+1]:.1f}: n={msk.sum():4d}  pred={p[msk].mean():.3f}  actual={y[msk].mean():.3f}")

Midfielder matches 2025-26: 4965  |  hit rate 0.090

Midfielder fix comparison (test GW21+):

fix             Brier  mean_pred  mean_actual
0_static       0.0608      0.102        0.065
3_walkfwd      0.0587      0.095        0.065
4_recency      0.0588      0.089        0.065

Baseline Brier (always base rate): 0.0630

Walk-forward calibration:
  0.0-0.1: n=1675  pred=0.036  actual=0.028
  0.1-0.2: n= 297  pred=0.141  actual=0.101
  0.2-0.3: n= 132  pred=0.249  actual=0.152
  0.3-0.4: n=  99  pred=0.353  actual=0.293
  0.4-0.5: n=  73  pred=0.442  actual=0.178
  0.5-0.6: n=  18  pred=0.554  actual=0.500


Attaching the relevant columns

In [44]:
# === Attach team, build rolling team-defensive-style feature ===
# Team lives in the gameweek ref file (per player per season). Get player_id -> team.
tcols = [c for c in gwref.columns if c in ["id","team","team_name","season"]]
print("Team-ish columns in gwref:", tcols)

# Build (season, player_id) -> team map. Team may be an id or name.
team_col = "team" if "team" in gwref.columns else ("team_name" if "team_name" in gwref.columns else None)
print("Using team column:", team_col)

tmap = gwref.dropna(subset=[team_col]).drop_duplicates(["season","id"]).set_index(["season","id"])[team_col]
d["team"] = d.set_index(["season","player_id"]).index.map(tmap)
print("Rows with a team:", d["team"].notna().sum(), "of", len(d))

Team-ish columns in gwref: ['id', 'team', 'season']
Using team column: team
Rows with a team: 11199 of 22766


In [45]:
# Confirm team coverage is complete for 2025-26 (the season we model)
cov25 = d[d["season"]=="2025-2026"]["team"].notna().mean()
print(f"2025-26 rows with team: {cov25:.1%}")

# === Build rolling TEAM defensive-volume feature ===
# Team defensive actions per match = sum of all players' CBIRT-components for that team-match.
# Use total defensive actions (cbirt sum) per team per gameweek.
team_gw = (d[d["season"]=="2025-2026"]
           .groupby(["team","gw"])
           .agg(team_def=("dc_metric","sum"), team_players=("dc_metric","size"))
           .reset_index())
team_gw["team_def_pp"] = team_gw["team_def"] / team_gw["team_players"]  # per player, normalizes squad size

# Rolling team defensive volume over PRIOR gameweeks (shift to avoid leakage)
team_gw = team_gw.sort_values(["team","gw"])
team_gw["team_def_roll"] = (team_gw.groupby("team")["team_def_pp"]
                            .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean()))

# Attach back to player-match rows
d = d.merge(team_gw[["team","gw","team_def_roll"]], on=["team","gw"], how="left")

print("Sample — teams by rolling defensive volume (higher = defends more):")
latest = team_gw[team_gw["gw"]==team_gw["gw"].max()].sort_values("team_def_roll", ascending=False)
print(latest[["team","team_def_roll"]].head(8).to_string(index=False))
print("...")
print(latest[["team","team_def_roll"]].tail(4).to_string(index=False))

2025-26 rows with team: 100.0%
Sample — teams by rolling defensive volume (higher = defends more):
          team  team_def_roll
     Brentford       5.657359
       Man Utd       5.443297
   Bournemouth       5.343780
         Leeds       5.083869
Crystal Palace       5.060977
       Everton       4.894872
      West Ham       4.806605
 Nott'm Forest       4.762857
...
       team  team_def_roll
    Arsenal       4.029524
    Chelsea       3.873663
     Fulham       3.709167
Aston Villa       2.985348


Checking if a particular team is more defensive helps the cause or not

In [46]:
# === Re-test midfielder model WITH the team-defensive-style feature ===
feat_base = ["roll_dc90_3c","roll_dc90_5c","roll_hit_5","roll_mins_3"]
feat_team = feat_base + ["team_def_roll"]

mids25 = d[(d["position"]=="Midfielder") & (d["season"]=="2025-2026")].dropna(subset=feat_team).copy().sort_values("gw")
test_gws = sorted([g for g in mids25["gw"].unique() if g >= 21])

def run(features):
    ys, ps = [], []
    for g in test_gws:
        te = mids25[mids25["gw"]==g]; prior = mids25[mids25["gw"]<g]
        if len(prior) < 200: continue
        m = mk().fit(prior[features], prior["dc_hit"])
        ys += list(te["dc_hit"]); ps += list(m.predict_proba(te[features])[:,1])
    return np.array(ys), np.array(ps)

print("Midfielder model — walk-forward, with vs without team feature:\n")
for label, features in [("base (no team)", feat_base), ("+ team_def_roll", feat_team)]:
    y, p = run(features)
    print(f"  {label:16s}: Brier {brier_score_loss(y,p):.4f}   mean_pred {p.mean():.3f} (actual {y.mean():.3f})")

Midfielder model — walk-forward, with vs without team feature:

  base (no team)  : Brier 0.0587   mean_pred 0.095 (actual 0.065)
  + team_def_roll : Brier 0.0588   mean_pred 0.096 (actual 0.065)


Bin by Bin view

In [47]:
# === Bin-by-bin: does the team feature fix the erratic middle bins? ===
def binview(y, p, label):
    edges = np.arange(0,1.1,0.1); idx = np.clip(np.digitize(p,edges)-1,0,9)
    print(f"\n{label}:")
    print(f"  {'bin':10s} {'n':>5s} {'pred':>7s} {'actual':>8s}")
    for b in range(10):
        msk = idx==b
        if msk.sum()<10: continue
        print(f"  {edges[b]:.1f}-{edges[b+1]:.1f}   {msk.sum():>5d} {p[msk].mean():>7.3f} {y[msk].mean():>8.3f}")

yb, pb = run(feat_base)
yt, pt = run(feat_team)
binview(yb, pb, "BASE (no team)")
binview(yt, pt, "+ team_def_roll")


BASE (no team):
  bin            n    pred   actual
  0.0-0.1    1675   0.036    0.028
  0.1-0.2     297   0.141    0.101
  0.2-0.3     132   0.249    0.152
  0.3-0.4      99   0.353    0.293
  0.4-0.5      73   0.442    0.178
  0.5-0.6      18   0.554    0.500

+ team_def_roll:
  bin            n    pred   actual
  0.0-0.1    1664   0.036    0.027
  0.1-0.2     298   0.141    0.117
  0.2-0.3     141   0.249    0.135
  0.3-0.4     110   0.347    0.218
  0.4-0.5      49   0.440    0.224
  0.5-0.6      27   0.539    0.370
  0.6-0.7      10   0.653    0.300


CDM only check

In [48]:
# === Does the model work for the DEFENSIVE mids (CDMs) specifically? ===
# Identify defensive mids empirically: high season-long CBIRT per 90.
# (No CDM label exists — we define it by defensive volume.)

mid_season = (d[(d["position"]=="Midfielder") & (d["season"]=="2025-2026")]
              .groupby("player_id")
              .agg(mins=("minutes_played","sum"), cbirt=("cbirt","sum"),
                   name=("player_id","first")).reset_index())
mid_season["name"] = mid_season["player_id"].map(gwref.drop_duplicates("id").set_index("id")["web_name"])
mid_season["cbirt90"] = mid_season["cbirt"]/mid_season["mins"]*90
mid_season = mid_season[mid_season["mins"]>=450]

# Top defensive mids by CBIRT/90 = our "CDM" group
cdm_ids = set(mid_season.nlargest(30, "cbirt90")["player_id"])
print("Top defensive mids identified (CDM group):")
print(mid_season.nlargest(12,"cbirt90")[["name","cbirt90"]].to_string(index=False))

# Now evaluate the walk-forward model ONLY on these CDMs
yb, pb = run(feat_base)   # re-run to get aligned predictions with player ids
# need player_id alongside predictions — rebuild with ids
def run_with_ids(features):
    rows=[]
    for g in test_gws:
        te = mids25[mids25["gw"]==g]; prior = mids25[mids25["gw"]<g]
        if len(prior)<200: continue
        m = mk().fit(prior[features], prior["dc_hit"])
        te=te.copy(); te["pred"]=m.predict_proba(te[features])[:,1]
        rows.append(te[["player_id","dc_hit","pred"]])
    return pd.concat(rows)

pred_df = run_with_ids(feat_base)
cdm = pred_df[pred_df["player_id"].isin(cdm_ids)]
other = pred_df[~pred_df["player_id"].isin(cdm_ids)]

print(f"\nModel performance split by group:")
print(f"  CDM group   (n={len(cdm):4d}): Brier {brier_score_loss(cdm['dc_hit'],cdm['pred']):.4f}  "
      f"mean_pred {cdm['pred'].mean():.3f}  actual {cdm['dc_hit'].mean():.3f}")
print(f"  Other mids  (n={len(other):4d}): Brier {brier_score_loss(other['dc_hit'],other['pred']):.4f}  "
      f"mean_pred {other['pred'].mean():.3f}  actual {other['dc_hit'].mean():.3f}")

# Calibration bins for CDMs only
print("\nCDM-only calibration:")
y,p = cdm["dc_hit"].values, cdm["pred"].values
edges=np.arange(0,1.1,0.1); idx=np.clip(np.digitize(p,edges)-1,0,9)
for b in range(10):
    msk=idx==b
    if msk.sum()<10: continue
    print(f"  {edges[b]:.1f}-{edges[b+1]:.1f}: n={msk.sum():4d}  pred={p[msk].mean():.3f}  actual={y[msk].mean():.3f}")

Top defensive mids identified (CDM group):
      name   cbirt90
      Cook 14.759174
    Ugarte 13.995460
  Anderson 12.957409
Florentino 12.741477
     Gomes 12.317131
     André 12.285276
 Bentancur 12.225159
   Wieffer 11.933962
J.Palhinha 11.776199
N.Gonzalez 11.544352
     Scott 11.364113
   Rodrigo 11.361533

Model performance split by group:
  CDM group   (n= 345): Brier 0.1813  mean_pred 0.236  actual 0.241
  Other mids  (n=1958): Brier 0.0371  mean_pred 0.070  actual 0.034

CDM-only calibration:
  0.0-0.1: n=  89  pred=0.049  actual=0.112
  0.1-0.2: n=  74  pred=0.145  actual=0.270
  0.2-0.3: n=  65  pred=0.253  actual=0.169
  0.3-0.4: n=  55  pred=0.355  actual=0.418
  0.4-0.5: n=  44  pred=0.448  actual=0.250
  0.5-0.6: n=  11  pred=0.556  actual=0.545


CBIRT for forwards >= 12

In [52]:
# === Forward DC (CBIRT >= 12) — is a model worth it, or is a rolling rate enough? ===
feat = ["roll_dc90_3c","roll_dc90_5c","roll_hit_5","roll_mins_3"]

fwd25 = d[(d["position"]=="Forward") & (d["season"]=="2025-2026")].dropna(subset=feat).copy().sort_values("gw")
print(f"Forward matches 2025-26: {len(fwd25)}  |  hit rate {fwd25['dc_hit'].mean():.3f}")

test_gws = sorted([g for g in fwd25["gw"].unique() if g >= 21])

# Compare three things: baseline (base rate), simple rolling-rate predictor, LGBM model
res = {"baseline":[], "rolling_rate":[], "lgbm":[]}
ys = []
for g in test_gws:
    te = fwd25[fwd25["gw"]==g]; prior = fwd25[fwd25["gw"]<g]
    if len(prior)<100 or len(te)==0: continue
    ys += list(te["dc_hit"])
    # baseline: prior hit rate for everyone
    res["baseline"] += [prior["dc_hit"].mean()]*len(te)
    # simple rolling-rate: use the player's own recent hit rate as the prediction
    res["rolling_rate"] += list(te["roll_hit_5"].values)
    # lgbm
    m = mk().fit(prior[feat], prior["dc_hit"])
    res["lgbm"] += list(m.predict_proba(te[feat])[:,1])

ys = np.array(ys)
print(f"\nEvaluated on {len(ys)} forward-matches (GW21+):\n")
for k, p in res.items():
    p = np.array(p)
    print(f"  {k:14s}: Brier {brier_score_loss(ys, p):.4f}  mean_pred {p.mean():.3f}  actual {ys.mean():.3f}")

Forward matches 2025-26: 1307  |  hit rate 0.004

Evaluated on 653 forward-matches (GW21+):

  baseline      : Brier 0.0031  mean_pred 0.005  actual 0.003
  rolling_rate  : Brier 0.0040  mean_pred 0.006  actual 0.003
  lgbm          : Brier 0.0041  mean_pred 0.003  actual 0.003
